In [1]:
%matplotlib inline

import matplotlib.pyplot as plt

import seaborn as sns

sns.set_theme()
plt.rcParams["figure.figsize"] = 9, 4.51

# Extractors Tutorial

## 1. Introduction

While `feets` comes with a wide array of pre-defined features, one of its core design principles is extensibility. You might have a novel feature in mind, need to implement an algorithm from a recent paper, or have a domain-specific metric that is not part of the standard library. `feets` allows you to seamlessly create and integrate your own feature extractors into its ecosystem.

This tutorial will guide you through the process of building custom feature extractors. You will learn how to define an extractor, register it with the library, handle dependencies, accept parameters, and even generate features with dynamic names.


## 2. Fundamentals


In `feets`, a feature extractor is a Python class responsible for calculating one or more features from a light curve. To create a valid extractor, you must follow these rules:

-   The class must inherit from `feets.Extractor`.
-   The class must define a `features` attribute, a list of strings containing the names of the features it's able to compute.
-   The class must implement an `extract()` method. This method implements the feature extraction logic. It receives the light curve data (e.g., `magnitude`, `time`, `error`) as parameters and returns a dictionary where keys are the feature names (from the `features` list) and values are the calculated feature values.

### Example 1: The `MaxMagMinTime` extractor

As an example, let's create a simple extractor that finds the maximum magnitude and the minimum time value in a light curve:


In [2]:
import feets

import numpy as np


# 1. Inherit from feets.Extractor
class MaxMagMinTime(feets.Extractor):
    # 2. Define the names of the features to be extracted
    features = ["magmax", "mintime"]

    # 3. Implement the extraction logic
    # The parameters are the data vectors of the light curve.
    def extract(self, magnitude, time):
        # The return value must be a dictionary with keys matching
        # the `features` list.
        return {"magmax": np.max(magnitude), "mintime": np.min(time)}

## 2. Registering an extractor

Once you have defined your extractor class, you need to add it to the underlying extractor registry in `feets` to make it available to use in any `FeatureSpace`. This is done using the `extractor_registry` utility:

In [3]:
from feets import extractor_registry

# Register the class to make it available in the FeatureSpace
extractor_registry.register_extractor(MaxMagMinTime)

__main__.MaxMagMinTime

Now, you can use the features `"magmax"` and `"mintime"` when defining a `FeatureSpace`, and `feets` will know how to compute them:

In [4]:
# Create a FeatureSpace using our new features
fs = feets.FeatureSpace(only={"magmax", "mintime"})

# Let's extract the features from some sample data
time = [1, 2, 3, 4, 5]
magnitude = [10.2, 10.5, 10.1, 10.6, 10.4]

features = fs.extract(time=time, magnitude=magnitude)
features.as_frame()

Features,mintime,magmax
Light Curve,,
0,1.0,10.6


## 3. Extractors with dependencies

With the `MaxMagMinTime` example, we've seen an extractor that computes features based on the provided light-curve data vectors. In addition to this data vectors, an extractor may also depend on features computed by other extractors.

To define a feature dependency, simply add the required feature name as a parameter to your extractor's `extract()` method, along with the other data vectors. Make sure that the dependency is computed by some extractor present in the extractor registry.

### Example 2: The `TimeDuration` extractor

Let's create an extractor that calculates the total duration of the light curve (`max_time - min_time`). We can reuse the `mintime` feature from our previous `MaxMagMinTime` extractor as a dependency for our `extract()` method:

In [5]:
class TimeDuration(feets.Extractor):
    features = ["duration"]

    # Add `mintime` as a parameter to define it as a dependency.
    def extract(self, time, mintime):
        return {"duration": np.max(time) - mintime}

# Register the new extractor
extractor_registry.register_extractor(TimeDuration)

__main__.TimeDuration

When you use this new extractor, `feets` will automatically resolve its dependencies. It will inspect the signature of the `extract()` method and identifie which arguments correspond to feature names.

In our `TimeDuration` example, `feets` recognizes that `mintime` is a required feature. It then searches the registry for an extractor that provides this feature (in this case, `MaxMagMinTime`) and ensures it is executed first. The resulting value is then passed as an argument to `TimeDuration.extract()`.

This dependency resolution happens behind the scenes when a `FeatureSpace` object is initialized, creating an efficient execution plan for the selected features. You only need to request the final features of interest, and feets will automatically include and execute all the necessary intermediate steps.

In [6]:
# When we ask for "duration", feets knows it needs "mintime" first.
# It will automatically add the MaxMagMinTime extractor to the plan.
fs = feets.FeatureSpace(only={"duration"})
print(f"Selected extractors in execution order: {fs.extractors}")

# The result will include both the requested and dependency features.
features = fs.extract(time=time, magnitude=magnitude)
features.as_frame()

Selected extractors in execution order: [MaxMagMinTime() TimeDuration()]


Features,duration
Light Curve,
0,4


## 4. Configure extractors with parameters

Some feature extraction algorithms require parameters to be configured, such as a specific quantile value or the number of bins for a histogram. `feets` supports this by allowing you to create configurable extractors.

To add parameters to your extractor, define an `__init__()` method in the class. This method can accept and store any parameters you need. These stored parameters are then available within the `extract()` method, allowing you to customize the feature calculation logic.

### Example 3: The `QuantileMagnitude` extractor

Let's illustrate this with an extractor that calculates the magnitude at a given quantile. The quantile itself will be a parameter that we can configure when we use the extractor:


In [7]:
class QuantileMagnitude(feets.Extractor):
    features = ["quantile_mag"]

    def __init__(self, quantile=0.5):
        # Store the parameter
        self.quantile = quantile

    def extract(self, magnitude):
        # Use the parameter in the calculation
        q_mag = np.quantile(magnitude, self.quantile)
        return {"quantile_mag": q_mag}

extractor_registry.register_extractor(QuantileMagnitude)

__main__.QuantileMagnitude

To use the new `QuantileMagnitude` extractor with a custom parameter, you need to specify the parameter's value when the `FeatureSpace` is initialized. This is achieved by passing a keyword argument to the `FeatureSpace` constructor where:

-   The keyword's name matches the extractor's class name (`QuantileMagnitude`).
-   The keyword's value is a dictionary containing the parameters to be passed to the extractor's `__init__()` method.

For instance, to compute the 90th percentile of the magnitude, you would configure the `QuantileMagnitude` extractor by setting its `quantile` parameter to `0.9`:

In [8]:
fs_quantile = feets.FeatureSpace(
    only=["quantile_mag"],
    QuantileMagnitude={"quantile": 0.9}
)

features = fs_quantile.extract(magnitude=magnitude)
features.as_frame()

Features,quantile_mag
Light Curve,
0,10.56


## 5. Handling multi-value features

Some feature extraction tasks naturally produce multiple values from a single computation. A common example is fitting a model to the data, where the result is a set of coefficients. `feets` is designed to handle this scenario gracefully.

The standard practice is to create an extractor that returns a single feature, but with its value being an array or a dictionary containing all the individual results. When you convert the extracted features to a `pandas.DataFrame` using the `.as_frame()` method of the resulting `Features` object, it will automatically "flatten" this non-scalar feature. It creates a separate column for each value in the array, this is tipically done appending a suffix to the original feature name.

### Example 4: The `PolynomialFit` Extractor

To demonstrate this, let's build an extractor that performs a polynomial fit on the light-curve data. This extractor will be configurable, allowing the user to specify the degree of the polynomial. The `extract()` method will compute the coefficients of the fit and return them as a single array-like feature.

In [9]:
class PolynomialFit(feets.Extractor):
  # Define the single, multi-value feature name.
  features = ["poly_coeffs"]

  def __init__(self, degree=1):
    super().__init__()
    self.degree = degree

  def extract(self, time, magnitude):
    # Fit the polynomial
    coeffs = np.polyfit(time, magnitude, self.degree)
    # Return the coefficients as a single array
    return {"poly_coeffs": coeffs}

extractor_registry.register_extractor(PolynomialFit)

__main__.PolynomialFit

In [10]:
# Instantiate the extractor with its parameters
fs_poly = feets.FeatureSpace(
  only=["poly_coeffs"],
  PolynomialFit={"degree": 2},
)

# The resulting feature set should have columns like `poly_coeffs_0`,
# `poly_coeffs_1`, etc.
features = fs_poly.extract(time=time, magnitude=magnitude)
features.as_frame()

Features,poly_coeffs_0,poly_coeffs_1,poly_coeffs_2
Light Curve,,,
0,-0.007143,0.092857,10.16


## 6. Customizing feature flattening

While the automatic flattening of multi-value features is convenient, there are times when you might want more control over the resulting column names or the flattening logic itself. `feets` allows you to customize this behavior by implementing a special `flatten_feature()` method in your extractor.

This method is called by `.as_frame()` when it encounters a non-scalar feature value from that extractor. It gives you the opportunity to define exactly how the feature should be represented in the final `pandas.DataFrame`.

The `flatten_feature()` method receives the feature name and its computed value and must return a dictionary where keys are the desired column names and values are the corresponding scalar values. This provides fine-grained control over the output for array-like and dictionary-like features and can even be used to handle more complex, custom data structures.

### Example 5: The `MagnitudeStats` extractor

Let's create an extractor called `MagnitudeStats` that computes several descriptive statistics (mean, standard deviation, and median) for the magnitude. Instead of defining a separate feature for each statistic, we will return them all within a single dictionary.

Then, we will implement the `flatten_feature()` method to transform this dictionary into a set of columns with custom names (e.g., `mag_mean`, `mag_std`). This approach keeps the feature extraction logic organized while providing full control over the final data representation:

In [11]:
class MagnitudeStats(feets.Extractor):
    # This extractor computes multiple stats and returns them as a dictionary.
    features = ["mag_stats"]

    def extract(self, magnitude):
        stats = {
            "mean": np.mean(magnitude),
            "std": np.std(magnitude),
            "median": np.median(magnitude),
        }
        return {"mag_stats": stats}

    # Implement the custom flattening logic.
    def flatten_feature(self, feature_name, feature_value):
        if feature_name != "mag_stats":
            # For features other than "mag_stats", use the default behavior.
            return super().flatten_feature(feature_name, feature_value)

        # For "mag_stats", we expect feature_value to be a dictionary.
        # We will prepend "mag_" to each stat name to create the column names.
        return {f"mag_{k}": v for k, v in feature_value.items()}


# Register the new extractor
extractor_registry.register_extractor(MagnitudeStats)

__main__.MagnitudeStats

In [12]:
# Create a FeatureSpace with our new feature
fs_stats = feets.FeatureSpace(only=["mag_stats"])

# Extract features and display the flattened DataFrame
# The columns will be "mag_mean", "mag_std", and "mag_median"
# thanks to our custom flatten_feature method.
features = fs_stats.extract(magnitude=magnitude)
features.as_frame()

Features,mag_mean,mag_std,mag_median
Light Curve,,,
0,10.36,0.185472,10.4


In [13]:
extractor_registry.unregister_extractor(TimeDuration)
extractor_registry.unregister_extractor(MaxMagMinTime)
extractor_registry.unregister_extractor(QuantileMagnitude)
extractor_registry.unregister_extractor(PolynomialFit)
extractor_registry.unregister_extractor(MagnitudeStats)